Please create an issue if anything in this colab breaks! :)

First lets clone the repo, we also require this repo: https://github.com/robflynnyh/long-context-asr and its dependencies as the pretrained model is from here, so we will clone and install this and any dependencies aswell!

In [ ]:
!git clone https://github.com/robflynnyh/Self-Train-Before-You-Transcribe
%cd Self-Train-Before-You-Transcribe
!git pull
%cd ..
!git clone https://github.com/robflynnyh/long-context-asr/
%cd long-context-asr/
!git checkout "v1.0" # long-context-asr is at release 1.0 at time of writing Self-Train paper
!pip install .
%cd ..
!ls

In [ ]:
!pip install einops
!pip install omegaconf
!pip install torch_ema

In [ ]:
%cd Self-Train-Before-You-Transcribe

/content/Self-Train-Before-You-Transcribe


In [ ]:
import lcasr
from main import dynamic_eval
import torch

fused_dense_cuda not available. Install from https://github.com/Dao-AILab/flash-attention/tree/main/csrc/fused_dense_lib for better performance!
flash attention not installed. Install from: github.com/Dao-AILab/flash-attention for best performance!


All modules are now installed and imported lets download the pretrained checkpoint and load a model for adaptation!

In [ ]:
!wget https://huggingface.co/rjflynn2/lcasr-6L-768D-6H-RB-1p5M/resolve/main/n_seq_sched_16384_rp_1/step_105360.pt # get model from huggingface

--2024-07-04 14:53:54--  https://huggingface.co/rjflynn2/lcasr-6L-768D-6H-RB-1p5M/resolve/main/n_seq_sched_16384_rp_1/step_105360.pt
Resolving huggingface.co (huggingface.co)... 3.163.189.114, 3.163.189.37, 3.163.189.90, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.114|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.huggingface.co/repos/ab/41/ab41cf0dd90aceaa7e6d95e0f06202a58e01294d14ca043be79b37cd2926f090/5802644ff521057ccbc92594ab97b793226b91851637fe289563638302c24177?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27step_105360.pt%3B+filename%3D%22step_105360.pt%22%3B&Expires=1720364035&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcyMDM2NDAzNX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmh1Z2dpbmdmYWNlLmNvL3JlcG9zL2FiLzQxL2FiNDFjZjBkZDkwYWNlYWE3ZTZkOTVlMGYwNjIwMmE1OGUwMTI5NGQxNGNhMDQzYmU3OWIzN2NkMjkyNmYwOTAvNTgwMjY0NGZmNTIxMDU3Y2NiYzkyNTk0YWI5N2I3OTMyMjZiOTE4N

In [ ]:
checkpoint = torch.load("step_105360.pt", map_location='cpu')
tokenizer = lcasr.utils.audio_tools.load_tokenizer()
model = lcasr.utils.general.load_model(
    config = checkpoint['config'],
    vocab_size = tokenizer.vocab_size(),
    model_class = lcasr.utils.general.get_model_class({'model_class': checkpoint['config'].get('model_class', 'SCConformerXL')})
)
model.load_state_dict(checkpoint['model'])
model.eval()
p = model.print_total_params()

Total params: :  89.914112 M


Done, now lets download some audio for adaptation and transcription! We will use a recording from earnings-22!

In [ ]:
!wget https://media.githubusercontent.com/media/revdotcom/speech-datasets/main/earnings22/media/4329526.mp3
!ls 4329526.mp3

--2024-07-04 14:54:15--  https://media.githubusercontent.com/media/revdotcom/speech-datasets/main/earnings22/media/4329526.mp3
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8414450 (8.0M) [audio/mpeg]
Saving to: ‘4329526.mp3.1’

4329526.mp3.1       100%[===================>]   8.02M  --.-KB/s    in 0.06s   

2024-07-04 14:54:16 (126 MB/s) - ‘4329526.mp3.1’ saved [8414450/8414450]

4329526.mp3


In [ ]:
spectrogram = lcasr.utils.audio_tools.processing_chain(path_in = './4329526.mp3', normalise=True)
print(spectrogram.shape)

torch.Size([1, 80, 210329])


In [ ]:
seq_len = checkpoint['config']['sequence_scheduler']['max_sequence_length']
print(f'Model has a sequence length of : {seq_len}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.device = device # store device on model namespace
print(f'Using device: {device}')

args = lcasr.utils.general.argsclass(
    config = checkpoint['config'],
    optim_lr = 9e-5, # 4.3 of paper (https://arxiv.org/pdf/2406.12937v1)
    spec_augment_n_time_masks = 0,
    spec_augment_zero_masking = False,
    spec_augment_n_freq_masks = 6,
    spec_augment_freq_mask_param = 34,
    epochs = 1, # 5 is used in paper as default setting, so use 5 for best results!
    shuffle = True,
    lm_tta_beams = 0,
)

logits = dynamic_eval(
  args,
  model.to(model.device),
  spectrogram.to(model.device),
  seq_len = seq_len,
  overlap = round(seq_len*0.875), # overlap ratio = (1 - stride ratio). We set the stride/overlap as a ratio of the sequence length. If confused see https://arxiv.org/abs/2310.15672v2
  tokenizer = tokenizer,
  beam_search_fn = False
)

Model has a sequence length of : 16384
Using device: cuda
{'n_time_masks': 0, 'n_freq_masks': 6, 'freq_mask_param': 34, 'time_mask_param': -1, 'min_p': 0.05, 'zero_masking': False} {'lr': 9e-05} {'time_dimension': False, 'freq_dimension': False} {'seq_len': 16384, 'cutout_val': 'mean', 'num_rectangles': 0, 'max_width': 100, 'max_height': 10}
Using seq_len: 16384 and overlap: 14336
Epoch 1 / 1


  0%|          | 0/96 [00:00<?, ?it/s]

torch.Size([1, 80, 16384])
Pseudo targets: hylo this is just to clarify we are not taking out capacity what i was trying to explain is that incorporation of the 14 air cupft happened around me next last year and therefore what you will see is the high growth in the first half as compared of the first half of last year because of the lower base and that gap will start torink as the year continues because we will have already the impact of those 14 airraft in we had it in the second semester of 2019. so capacity will still be positive and our guidanceances 79 percent for the secret in the year as a whole. okay. so you have impro as i say as compared to last year in the first quarter and the comparison the second quarter will feel see and also how the market develops the only point of consideration is that yield base also increa significantly last year because by the time of the second quarter ofianca has mostly receive its overation. yeah, so that was my concern because you know, there w

  1%|          | 1/96 [00:03<05:35,  3.53s/it]

torch.Size([1, 80, 16384])
Pseudo targets: we so a decline of zero 7% point in load fact of56.4 percent reish fork decined by 8% in the fourth quarter mainly doing by lower import into region especially to brazil and argentina and disruptions del with by the social breath during the fourth quarter of 2019. please turn to slide number five once again, and from this slide so you can practice the maintain hisike point of sale of the co and kind revenues in the past 12 months. on let's try you can see the reies by point sa in 2018 here butil represent approimately 35% of totald revense while atentina december 10er% but revenues. if you move to be drafted right which would like done by country after the same period in 2019. we can see a 4 points increase for brail while argentina declined four points account for all six% at the point of sale. thisversification allows us to ad suggest ourations and to set the impacts that we maytain in certain markets. please turn now to slide them mistakes 

...

 98%|█████████▊| 94/96 [02:04<00:02,  1.34s/it]

torch.Size([1, 80, 16384])
Pseudo targets: million passengers the highest numbers in our history and more than 5 million additional passengers half of them in brazil compared to8. which represented more growth than any other airlines in the region. in addition to growth we continue proving two main periods where we put a special attention our operations and our customers. regarding our operations. we improve the capital capity of our network, especially from our hu in saniago lima and some powder at the same time. we put the special focus on their reliability of our operations. we are proud to be named the most functionual airline in the world in the megaary category by a ohg and byera we approximately 86 percent of our flights being gone time. in 2019. we have also been recogn recognized as this most functionual airline in brazil. cans exec fu remains stles at 4 point5ents despite challenges and the effects of the social invest in the region in particular in jila. growing in shorter r

 99%|█████████▉| 95/96 [02:06<00:01,  1.34s/it]

torch.Size([1, 80, 16384])
Pseudo targets: where we put a special attention our operations and our customers. regarding our operations. we improve the capital capity of our network, especially from our hu in saniago lima and some powder at the same time. we put the special focus on their reliability of our operations. we are proud to be named the most functionual airline in the world in the megaary category by a ohg and byera we approximately 86 percent of our flights being gone time. in 2019. we have also been recognized as this most functional airline in brazil. cans exec fu remains stable at 4 point5ents despite challenges and the effects of the social invest in the region in particular angila. growing in shorter roots and cutting capacity longer routots mainly those from argentina early in in the year due to thealuation of the schedule reduced our average stage length by 3 percent for the year the developing fix costff per flight in fuer head states. in addition the c grounding of 

100%|██████████| 96/96 [00:22<00:00,  4.18it/s]


We need now need to decode the logits from CTC logprobs to text, we will use greedy decoding!

In [ ]:
print(logits.shape)
from lcasr.decoding.greedy import GreedyCTCDecoder
decoder = GreedyCTCDecoder(tokenizer = tokenizer, blank_id = model.decoder.num_classes-1)
out_text = decoder(torch.as_tensor(logits))

(26292, 4096)


In [ ]:
print('Model output:\n',out_text.replace(". ", ".\n"))

Model output:
 today everyone and welcome to latan airlines group airning release conference call just as a reminder this conferenceere is being recorded lat time airlines group earnings release for the period was distributed on tuesday, march 3rd.
if you have not received it, you can find it in our website at www..ines group dotnet in investor relation section at this time.
i would like to point out that statements regarding the company's business outlook and anticipated financial and operating results constitute forward-looking comments.
this expectations are highly dependent on the economy the airline industry and the international markets.
therefore they are subject to change now 8 is my pleasure to turn the call over to mr.
amido al on seen chief financial officer of lat time airlines group.
mr.
alson seen, please begin.
thank you carmen and good morning, everyone and welcome to lat air line fourth quarter earnings call joining me today i mr.
robert toaluief commercial officer.
mr

for reference lets transcribe without adaptation (i.e normal evaluation)

In [ ]:
from lcasr.eval.buffered_transcription import fetch_logits as buffered_eval

In [ ]:
seq_len = checkpoint['config']['sequence_scheduler']['max_sequence_length']
print(f'Model has a sequence length of : {seq_len}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.device = device # store device on model namespace
print(f'Using device: {device}')
logits = buffered_eval(
    args = lcasr.utils.general.argsclass(**{'config': checkpoint['config']}),
    model = model.to(device),
    spec = spectrogram,
    seq_len = seq_len,
    overlap = round(seq_len*0.875), # overlap ratio = (1 - stride ratio). We set the stride/overlap as a ratio of the sequence length. If confused see paper :)
    tokenizer = tokenizer,
    use_tqdm = True
)
print(logits.shape, '!')

Model has a sequence length of : 16384
Using device: cuda
Using seq_len: 16384 and overlap: 14336


100%|██████████| 103/103 [00:21<00:00,  4.84it/s]


(26292, 4096) !


In [ ]:
decoder = GreedyCTCDecoder(tokenizer = tokenizer, blank_id = model.decoder.num_classes-1)
out_text_normal_eval = decoder(torch.as_tensor(logits))

In [ ]:
print('Model output:\n',out_text_normal_eval.replace(". ", ".\n"))

Model output:
 today everyone and welcome to latin airlines group earnings release conference call just as i reminder this confere is being recorded let time airlines group earnings release for the period was distributed on tuesday, march 3rd.
if you have not received it, you can find it in our website at www..ines group dotnet in the investor relation section at this time.
i would like to point out that statements regarding the company's business outlook and anticipated financial and operating results constitute forwardlooking comments.
this expectations are highly dependent on the economy the airline industry and the international markets.
therefore they are subject to change now.
it is my pleasure to turn the call over to mr.
raio alph seen chief financial officer of latan airlines group.
mr.
alph seen, please begin.
thank you commonmen and good morning everyone and welcome to the lines fourth quarter earnings callll joining me today i mr.
robert to alowief commercial officer.
mr.ad